In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nesmanasser/artworks2/artworks.json


In [3]:
!pip install -q \
transformers==4.52.4 \
langchain==0.3.27 \
langchain-core==0.3.74 \
langchain-community==0.3.27 \
langchain-classic \
langchain-huggingface==0.3.1 \
sentence-transformers \
faiss-cpu \
bitsandbytes \
accelerate

ERROR: Cannot install langchain-classic==1.0.0, langchain-classic==1.0.1, langchain-classic==1.0.2, langchain-classic==1.0.3, langchain-classic==1.0.4, langchain-classic==1.0.5, langchain-classic==1.0.6, langchain-classic==1.0.7, langchain-classic==1.0.8, langchain-community==0.3.27, langchain-core==0.3.74, langchain-huggingface==0.3.1 and langchain==0.3.27 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [4]:
!pip install -q bitsandbytes accelerate
!pip install -q transformers==4.52.4 sentence-transformers faiss-cpu
!pip install -q langchain==0.3.27 langchain-core==0.3.74 langchain-community==0.3.27 langchain-huggingface==0.3.1 langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 44.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 91.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 93.8 MB/s eta 0:00:00
ERROR: Cannot install langchain-classic==1.0.0, langchain-classic==1.0.1, langchain-classic==1.0.2, langchain-classic==1.0.3, langchain-classic==1.0.4, langchain-classic==1.0.5, langchain-classic==1.0.6, langchain-classic==1.0.7, langchain-classic==1.0.8, langchain-community==0.3.27, langchain-core==0.3.74, langchain-huggingface==0.3.1 and langchain==0.3.27 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [5]:
!pip install -q bitsandbytes accelerate
!pip install -q langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 34.5 MB/s eta 0:00:00


In [6]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
print("GPU متاحة:", torch.cuda.is_available())
print("اسم الكارت:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "مفيش")

GPU متاحة: True
اسم الكارت: Tesla T4


In [8]:
!pip list | grep langchain

langchain                                1.2.15
langchain-classic                        1.0.8
langchain-core                           1.5.2
langchain-protocol                       0.0.18
langchain-text-splitters                 1.1.2


In [9]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Mistral Loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Mistral Loaded


In [10]:
def generate_text(prompt, max_new_tokens=500):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[0][inputs.shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [11]:
prompt = """
You are a helpful assistant.

Question:
What is Artificial Intelligence?

Answer:
"""

print(generate_text(prompt))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Artificial Intelligence (AI) is a broad field of computer science dedicated to creating smart machines capable of performing tasks that typically require human intelligence. These tasks include learning (acquiring information and rules for using the information), reasoning (using the rules to reach approximate or definite conclusions), and problem-solving.

AI can be categorized into several types, including:

1. **Rule-Based AI**: This is the simplest form of AI, where a set of rules is defined, and the AI system follows these rules to make decisions or predictions.

2. **Machine Learning (ML)**: This is a subset of AI that involves training algorithms on data to make predictions or decisions without being explicitly programmed. It's further divided into:
   - Supervised Learning: The algorithm learns from labeled data, i.e., data with predefined outputs.
   - Unsupervised Learning: The algorithm learns from unlabeled data, finding patterns and relationships on its own.
   - Reinforce

In [12]:
from typing import Any
from langchain_core.language_models.llms import LLM

class CustomHFLLM(LLM):

    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

    def _call(self, prompt: str, stop: Any = None) -> str:
        return generate_text(prompt)

llm = CustomHFLLM()

print("✅ CustomHFLLM Ready")

✅ CustomHFLLM Ready


In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are a helpful assistant.

Question:
{question}

Answer:
"""
)

chain = LLMChain(
    llm=llm,
    prompt=prompt
)

response = chain.run(
    question="What is Machine Learning?"
)

print(response)

/tmp/ipykernel_59/569043523.py:16: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(
/tmp/ipykernel_59/569043523.py:21: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = chain.run(
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Machine Learning (ML) is a subset of artificial intelligence (AI) that involves training models on data to make predictions or decisions without being explicitly programmed. Here's a simple breakdown:

1. **Supervised Learning**: This is like learning with a teacher. You have input data (like student grades) and corresponding output data (like whether they passed or failed). The model learns to predict the output from the input. Examples include:
   - Classification: Predicting whether an email is spam or not (spam classifier).
   - Regression: Predicting a house's price based on its features (house price predictor).

2. **Unsupervised Learning**: This is like learning without a teacher. You only have input data, and the model tries to find patterns or structure on its own. Examples include:
   - Clustering: Grouping customers based on their purchasing behavior (customer segmentation).
   - Dimensionality Reduction: Reducing the number of features in data while retaining as much inform

In [14]:
from langchain_classic.output_parsers import (
    ResponseSchema,
    StructuredOutputParser
)

artwork_schema = ResponseSchema(
    name="artwork",
    description="The artwork title."
)

artist_schema = ResponseSchema(
    name="artist",
    description="The artist name."
)

year_schema = ResponseSchema(
    name="year",
    description="The creation year if available."
)

answer_schema = ResponseSchema(
    name="answer",
    description="""
The final museum-style answer.

It MUST follow the language requested in the prompt.

If the prompt requires Arabic,
the answer MUST be entirely in Arabic.

If the prompt requires English,
the answer MUST be entirely in English.

For general questions about the artwork,
write a complete explanation using all relevant information from the context,
including:
- story
- symbols
- historical context
- artist biography
- interesting facts

Do not invent information.

Do not omit important details.

Use only the provided context.
"""
)

response_schemas = [
    artwork_schema,
    artist_schema,
    year_schema,
    answer_schema
]

output_parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)

format_instructions = output_parser.get_format_instructions()

print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"artwork": string  // The artwork title.
	"artist": string  // The artist name.
	"year": string  // The creation year if available.
	"answer": string  // 
The final museum-style answer.

It MUST follow the language requested in the prompt.

If the prompt requires Arabic,
the answer MUST be entirely in Arabic.

If the prompt requires English,
the answer MUST be entirely in English.

For general questions about the artwork,
write a complete explanation using all relevant information from the context,
including:
- story
- symbols
- historical context
- artist biography
- interesting facts

Do not invent information.

Do not omit important details.

Use only the provided context.

}
```


In [15]:
import re
 
def detect_language(text: str) -> str:
    """
    بنحدد اللغة في بايثون (دقيق 100%) بدل ما نسيب Mistral يخمّن.
    ده أهم خطوة في حل المشكلة - الموديل مبيبقاش عنده مجال يغلط.
    """
    if re.search(r'[\u0600-\u06FF\u0750-\u077F]', text):
        return "arabic"
    return "english"
 
 
def build_language_instruction(language: str) -> str:
    if language == "arabic":
       return """
أجب باللغة العربية الفصحى فقط.

لا تستخدم أي كلمة إنجليزية.

لا تترجم أسماء اللوحات أو أسماء الفنانين.

لا تخترع أي معلومة.

استخدم المعلومات الموجودة في الـ Context فقط.
"""
    return "You must write the \"answer\" field entirely in English. Do not use any Arabic words in the answer field."
 

In [16]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are ArtMuse AI.

You are an expert museum guide at the Metropolitan Museum of Art.

Your ONLY source of knowledge is the provided Context.

====================================================
STRICT RULES
====================================================

1. NEVER use outside knowledge.

2. NEVER guess.

3. NEVER invent facts.

4. NEVER mention information that is not explicitly written in the Context.

5. If the requested information is missing from the Context, reply exactly:

"I couldn't find this information in the museum database."

6. Answer naturally like a museum guide.

7. Do NOT mention words such as:
"According to the context"
"The database says"

Just answer naturally.

====================================================
LANGUAGE
====================================================

{language_instruction}

The entire answer MUST be in that language.

====================================================
GENERAL QUESTIONS
====================================================

If the visitor asks a general question such as:

- Tell me about this artwork
- Tell me about this painting
- Describe this artwork
- احكيلي عن اللوحة
- احكيلي عن الرسمة
- كلمني عن اللوحة
- عرفني باللوحة

Then write a complete museum-style explanation.

Include ONLY the sections that actually exist inside the Context.

Possible sections include:

• Artwork Name
• Artist
• Year
• Story
• Symbols
• Historical Context
• Artist Biography
• Interesting Facts

Write them naturally as one coherent explanation.

Do NOT make up missing sections.

====================================================
SPECIFIC QUESTIONS
====================================================

If the visitor asks about ONE topic only, answer ONLY that topic.

Examples:

Who painted this?
→ Answer only the artist.

What do the symbols mean?
→ Answer only the Symbols section.

Interesting facts?
→ Answer only Interesting Facts.

Historical background?
→ Answer only Historical Context.

====================================================
CONTEXT
====================================================

{context}

====================================================
QUESTION
====================================================

{question}

====================================================
OUTPUT
====================================================

Return ONLY valid JSON.

{format_instructions}

""",
    input_variables=[
        "context",
        "question",
        "language_instruction"
    ],
    partial_variables={
        "format_instructions": format_instructions
    }
)

In [17]:
from langchain_classic.chains import LLMChain

chain = prompt | llm | output_parser

In [18]:
context = """
Artwork:
The Harvesters
 
Artist:
Pieter Bruegel the Elder
 
Year:
1565
 
Story:
The painting depicts peasants harvesting wheat during summer.
 
Historical Context:
It belongs to Bruegel's famous cycle of seasonal paintings.
"""
 
question = "احكيلي عن اللوحة"
 
parsed = chain.invoke({
    "context": context,
    "question": question,
    "language_instruction": build_language_instruction(detect_language(question)),
})
 
print(parsed)
 

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{'artwork': 'The Harvesters', 'artist': 'Pieter Bruegel the Elder', 'year': '1565', 'answer': "هذه اللوحة، entitled 'The Harvesters'، من عمل الفنان الفلمنكي العظيم بيتر بروغل الأكبر. تم إنشاؤها في عام 1565. تركز اللوحة على مشهد من حياة peasants، حيث يصورون في منتصف موسم الحصاد. يصور بروغل، المعروف بتمثيلاته التفصيلية للحياة اليومية، العمال في حقل من القمح، وهم يحرثون في حرارة الصيف. اللوحة جزء من سلسلة من الأعمال الموسمية الشهيرة لبروغل. إنها تثير إعجابنا بالتفاصيل الدقيقة والرسوم المعبرة، مما يوفر لنا نظرة ثاقبة إلى الحياة في عصره."}


In [19]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [20]:
from fastapi import FastAPI
from pydantic import BaseModel

import nest_asyncio
import uvicorn

from pyngrok import ngrok

In [21]:
app = FastAPI(title="ArtMuse AI API")

In [22]:
from fastapi import FastAPI
from pydantic import BaseModel
 
 
class QueryRequest(BaseModel):
    context: str
    question: str
 
 
@app.post("/generate")
def generate(request: QueryRequest):

    language = detect_language(request.question)
    language_instruction = build_language_instruction(language)

    print("=" * 60)
    print("Question:")
    print(request.question)

    print("\nDetected:")
    print(language)

    print("\nInstruction:")
    print(language_instruction)

    print("=" * 60)

    parsed = chain.invoke({
        "context": request.context,
        "question": request.question,
        "language_instruction": language_instruction,
    })

    print(parsed)

    return parsed

In [23]:
from pyngrok import ngrok

ngrok.set_auth_token("3GromvzkG1Pm7vXqKTzmetro5XC_2s6Xn7Nx3Wvffx8mFpmKk")

In [24]:
NGROK_TOKEN = "3GromvzkG1Pm7vXqKTzmetro5XC_2s6Xn7Nx3Wvffx8mFpmKk"

In [25]:
import threading
import time
import socket
import uvicorn
from pyngrok import ngrok, conf

# اختيار بورت فاضي
def free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()

# لو لسه معملتيش auth token
conf.get_default().auth_token = NGROK_TOKEN

# إنشاء الـ Tunnel
public_url = ngrok.connect(port).public_url

print("Public URL:", public_url)

# تشغيل FastAPI في Thread منفصل
def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=port,
        log_level="info"
    )

threading.Thread(target=run, daemon=True).start()

time.sleep(2)

print("✅ FastAPI is running!")

Public URL: https://tidal-easily-diligence.ngrok-free.dev


INFO:     Started server process [59]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:39019 (Press CTRL+C to quit)


✅ FastAPI is running!


In [26]:
import requests

url = "https://tidal-easily-diligence.ngrok-free.dev/generate"

headers = {
    "Authorization": "Bearer secret123"
}

payload = {
    "context": """
Artwork:
The Harvesters

Artist:
Pieter Bruegel the Elder

Year:
1565

Story:
The painting depicts peasants harvesting wheat during summer.

Historical Context:
It belongs to Bruegel's famous cycle of seasonal paintings.

Interesting Facts:
It is considered one of the greatest landscape paintings.
""",
    "question": "احكيلي عن اللوحة"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

print(response.status_code)
print(response.json())

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question:
احكيلي عن اللوحة

Detected:
arabic

Instruction:

أجب باللغة العربية الفصحى فقط.

لا تستخدم أي كلمة إنجليزية.

لا تترجم أسماء اللوحات أو أسماء الفنانين.

لا تخترع أي معلومة.

استخدم المعلومات الموجودة في الـ Context فقط.

{'artwork': 'The Harvesters', 'artist': 'Pieter Bruegel the Elder', 'year': '1565', 'answer': "هذه اللوحة، التي تحمل اسم 'الحراثون'، هي عمل من أعمال الفنان الفلمنكي العظيم بيتر بروغل الأكبر. تم إنشاؤها في عام 1565، وتعد جزءً من سلسلة بروغل الشهيرة التي تركز على المواسم. يصور العمل فلاحين يحصدون القمح في صيفٍ حارٍ. يرمز هذا العمل إلى حياة peasants، والعمالة الشاقة، والطبيعة، والاحتفال بالاحتفالات الزراعية. من بين الفاتنات المثيرة للاهتمام، فإن هذا العمل يُعد من أفضل أعمال اللوحة في عصره."}
INFO:     34.187.188.192:0 - "POST /generate HTTP/1.1" 200 OK
200
{'artwork': 'The Harvesters', 'artist': 'Pieter Bruegel the Elder', 'year': '1565', 'answer': "هذه اللوحة، التي تحمل اسم 'الحراثون'، هي عمل من أعمال الفنان الفلمنكي العظيم بيتر بروغل الأكبر. تم إنشاؤها في عام

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question:
What do the symbols in this painting mean?

Detected:
english

Instruction:
You must write the "answer" field entirely in English. Do not use any Arabic words in the answer field.
INFO:     156.218.59.46:0 - "POST /generate HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

...

Question:
Visitor: كلمني عن الرموز اللي في اللوحة
Visitor: ايه الرموز اللي في اللوحة
Visitor: Tell me the story behind this painting.

Detected:
arabic

Instruction:

أجب باللغة العربية الفصحى فقط.

لا تستخدم أي كلمة إنجليزية.

لا تترجم أسماء اللوحات أو أسماء الفنانين.

لا تخترع أي معلومة.

استخدم المعلومات الموجودة في الـ Context فقط.

{'artwork': 'The Lamentation', 'artist': 'Ambrosius Benson', 'year': 'ca. 1520–25', 'answer': 'مشهد تعبدي مهيب ومحرك للمشاعر يمثل النواح على جسد المسيح عقب إنزاله من الصليب؛ حيث يتوسط التكوين جسده الشاحب مستندًا على ركبتي العذراء المنهارة، بينما يسندها الرسول يوحنا بحنو، وتجلس مريم المجدلية عند الأقدام باكية بوعاء الطيب الخاص بها، في توليفة تدمج رقة الفن الشمالي مع الدراما الإيطالية. يرمز الكفن الكتاني الأبيض الساطع إلى طهارة الفداء وجسد الكنيسة الفتية، في حين يعكس وعاء الطيب في يد المجدلية رمزية الأمل في القيامة الوشيكة، ويمثل الشكل المقوس العلوي للوحة محاكاة بصرية لقبة السماء ومظلة العناية الإلهية المحيطة بالآلام الإنسانية.'}
INFO:     156.218.59.46:0

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question:
Visitor: كلمني عن الرموز اللي في اللوحة
Visitor: ايه الرموز اللي في اللوحة
Visitor: Tell me the story behind this painting.
Docent: مشهد تعبدي مهيب ومحرك للمشاعر يمثل النواح على جسد المسيح عقب إنزاله من الصليب؛ حيث يتوسط التكوين جسده الشاحب مستندًا على ركبتي العذراء المنهارة، بينما يسندها الرسول يوحنا بحنو، وتجلس مريم المجدلية عند الأقدام باكية بوعاء الطيب الخاص بها، في توليفة تدمج رقة الفن الشمالي مع الدراما الإيطالية. يرمز الكفن الكتاني الأبيض الساطع إلى طهارة الفداء وجسد الكنيسة الفتية، في حين يعكس وعاء الطيب في يد المجدلية رمزية الأمل في القيامة الوشيكة، ويمثل الشكل المقوس العلوي للوحة محاكاة بصرية لقبة السماء ومظلة العناية الإلهية المحيطة بالآلام الإنسانية.
Visitor: Tell me more about Petrus Christus.

Detected:
arabic

Instruction:

أجب باللغة العربية الفصحى فقط.

لا تستخدم أي كلمة إنجليزية.

لا تترجم أسماء اللوحات أو أسماء الفنانين.

لا تخترع أي معلومة.

استخدم المعلومات الموجودة في الـ Context فقط.

{'artwork': 'Head of Christ (Ecce Homo)', 'artist': 'Petrus Christus'

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question:
Visitor: كلمني عن الرموز اللي في اللوحة
Visitor: ايه الرموز اللي في اللوحة
Visitor: Tell me the story behind this painting.
Docent: مشهد تعبدي مهيب ومحرك للمشاعر يمثل النواح على جسد المسيح عقب إنزاله من الصليب؛ حيث يتوسط التكوين جسده الشاحب مستندًا على ركبتي العذراء المنهارة، بينما يسندها الرسول يوحنا بحنو، وتجلس مريم المجدلية عند الأقدام باكية بوعاء الطيب الخاص بها، في توليفة تدمج رقة الفن الشمالي مع الدراما الإيطالية. يرمز الكفن الكتاني الأبيض الساطع إلى طهارة الفداء وجسد الكنيسة الفتية، في حين يعكس وعاء الطيب في يد المجدلية رمزية الأمل في القيامة الوشيكة، ويمثل الشكل المقوس العلوي للوحة محاكاة بصرية لقبة السماء ومظلة العناية الإلهية المحيطة بالآلام الإنسانية.
Visitor: Tell me more about Petrus Christus.
Docent: لوحة 'رأس المسيح' (Ecce Homo) هي لوحة دينية صغيرة ومميزة، تبتعد عن السرد التاريخي وتقدم وجه المسيح المتألم بنظرة مباشرة وآسرة للمشاهد. يزاوج الفنان بعبقرية بين تاج الشوك النازف بالدم وحركة يده المرفوعة بالبركة الكلاسيكية، محفزًا طقوس التأمل الفردي العاطفي. يرمز تاج 